# Role-wise DECLARE Model Discovery

This notebook discovers DECLARE rules separately for each role-specific OCEL log in `role_logs/` and saves per-role rule files to `declare_models/`.


In [1]:
import os
import re
import json
from pathlib import Path

import pandas as pd
import pm4py
from IPython.display import display
from pm4py.util import constants
from pm4py.objects.log.obj import EventLog

from Declare4Py.D4PyEventLog import D4PyEventLog
from Declare4Py.ProcessMiningTasks.Discovery.DeclareMiner import DeclareMiner

LOG_DIR             = Path('role_logs')
OUTPUT_DIR          = Path('declare_models')
FLATTEN_OBJECT_TYPE = 'issue'
MIN_SUPPORT         = 0.5
ITEMSETS_SUPPORT    = 0.9
MAX_CARDINALITY     = 2
CONSIDER_VACUITY    = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

role_files = sorted([
    f for f in LOG_DIR.iterdir()
    if f.name.startswith('ocel_') and f.suffix == '.json'
])

print('Configuration:')
print(f'  Role log directory   : {LOG_DIR}')
print(f'  Output directory     : {OUTPUT_DIR}')
print(f'  Flatten object type  : {FLATTEN_OBJECT_TYPE}')
print(f'  Min support          : {MIN_SUPPORT}')
print(f'  Itemsets support     : {ITEMSETS_SUPPORT}')
print(f'  Max cardinality      : {MAX_CARDINALITY}')
print(f'  Consider vacuity     : {CONSIDER_VACUITY}')
print(f'  Role logs found      : {len(role_files)}')


Configuration:
  Role log directory   : role_logs
  Output directory     : declare_models
  Flatten object type  : issue
  Min support          : 0.5
  Itemsets support     : 0.9
  Max cardinality      : 2
  Consider vacuity     : False
  Role logs found      : 8


In [2]:
def ocel_to_d4py(path: str, object_type: str = FLATTEN_OBJECT_TYPE) -> tuple[D4PyEventLog, pd.DataFrame]:
    """Read an OCEL2 JSON, flatten on object_type, return (D4PyEventLog, flat_df)."""
    ocel = pm4py.read_ocel2_json(path)
    flat_df = pm4py.ocel_flattening(ocel, object_type)
    el: EventLog = pm4py.convert_to_event_log(flat_df)
    el._properties[constants.PARAMETER_CONSTANT_ACTIVITY_KEY] = 'concept:name'
    el._properties[constants.PARAMETER_CONSTANT_TIMESTAMP_KEY] = 'time:timestamp'
    return D4PyEventLog(log=el), flat_df


def parse_constraint(raw: str) -> dict:
    """Parse a Declare4Py constraint string like 'Response[A, B] | |'."""
    m = re.match(r'([^\[]+)\[([^\]]+)\]', raw.strip())
    if not m:
        return {'template': raw, 'activity_a': '', 'activity_b': ''}
    template = m.group(1).strip()
    activities = [a.strip() for a in m.group(2).split(',')]
    return {
        'template': template,
        'activity_a': activities[0] if len(activities) > 0 else '',
        'activity_b': activities[1] if len(activities) > 1 else '',
    }


def save_role_constraints(role: str, flat_df: pd.DataFrame, raw_constraints: list[str]) -> Path:
    """Save per-role DECLARE constraints as JSON."""
    output_path = OUTPUT_DIR / f'declare_{role}.json'
    payload = {
        'role': role,
        'flatten_object_type': FLATTEN_OBJECT_TYPE,
        'min_support': MIN_SUPPORT,
        'itemsets_support': ITEMSETS_SUPPORT,
        'max_cardinality': MAX_CARDINALITY,
        'consider_vacuity': CONSIDER_VACUITY,
        'cases': int(flat_df['case:concept:name'].nunique()) if 'case:concept:name' in flat_df.columns else 0,
        'events': int(len(flat_df)),
        'distinct_activities': int(flat_df['concept:name'].nunique()) if 'concept:name' in flat_df.columns else 0,
        'constraint_count': int(len(raw_constraints)),
        'constraints': [
            {
                'raw': raw,
                **parse_constraint(raw)
            }
            for raw in raw_constraints
        ]
    }
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    return output_path


In [3]:
print('Loading and flattening role logs ...')

d4py_roles: dict[str, D4PyEventLog] = {}
flat_roles: dict[str, pd.DataFrame] = {}
load_rows = []

for file_path in role_files:
    role = file_path.stem.replace('ocel_', '')
    d4py_log, flat_df = ocel_to_d4py(str(file_path))
    d4py_roles[role] = d4py_log
    flat_roles[role] = flat_df
    load_rows.append({
        'role': role,
        'cases': int(flat_df['case:concept:name'].nunique()) if 'case:concept:name' in flat_df.columns else 0,
        'events': int(len(flat_df)),
        'distinct_activities': int(flat_df['concept:name'].nunique()) if 'concept:name' in flat_df.columns else 0,
    })

print(f"\nFlattened role log summary (object type = '{FLATTEN_OBJECT_TYPE}'):")
display(pd.DataFrame(load_rows).set_index('role').sort_index())


Loading and flattening role logs ...


/Users/liorlimonad/miniforge3/lib/python3.10/site-packages/pm4py/util/dt_parsing/parser.py:77: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(



Flattened role log summary (object type = 'issue'):


,cases,events,distinct_activities
role,,,
bot,1045,3394,11
contributor,109,236,1
devops_engineer,7,15,1
feature_developer,27,100,1
issue_reporter,1436,16037,34
maintainer,837,2149,1
quality_engineer,60,194,1
technical_writer,37,71,1


In [4]:
declare_role_models: dict[str, object] = {}
discovery_rows = []

for role in sorted(flat_roles):
    flat = flat_roles[role]
    n_cases = int(flat['case:concept:name'].nunique()) if 'case:concept:name' in flat.columns else 0
    n_acts = int(flat['concept:name'].nunique()) if 'concept:name' in flat.columns else 0

    print(f"\n{'='*60}")
    print(f"ROLE: {role}")
    print(f"{'='*60}")
    print(f"Cases: {n_cases}")
    print(f"Distinct activities: {n_acts}")

    if flat.empty:
        print('Skipping: flattened log is empty')
        discovery_rows.append({
            'role': role,
            'cases': n_cases,
            'distinct_activities': n_acts,
            'constraints': 0,
            'status': 'skipped_empty',
            'output_file': None,
        })
        continue

    if n_acts < 2:
        print('Skipping: fewer than 2 distinct activities')
        discovery_rows.append({
            'role': role,
            'cases': n_cases,
            'distinct_activities': n_acts,
            'constraints': 0,
            'status': 'skipped_low_activity_diversity',
            'output_file': None,
        })
        continue

    miner = DeclareMiner(
        log=d4py_roles[role],
        consider_vacuity=CONSIDER_VACUITY,
        min_support=MIN_SUPPORT,
        itemsets_support=ITEMSETS_SUPPORT,
        max_declare_cardinality=MAX_CARDINALITY,
    )
    model = miner.run()
    declare_role_models[role] = model

    raw_constraints = model.get_decl_model_constraints()
    print(f'Discovered {len(raw_constraints)} constraints')

    output_path = save_role_constraints(role, flat, raw_constraints)
    print(f'Saved DECLARE rules to: {output_path}')

    discovery_rows.append({
        'role': role,
        'cases': n_cases,
        'distinct_activities': n_acts,
        'constraints': int(len(raw_constraints)),
        'status': 'saved',
        'output_file': str(output_path),
    })

    if raw_constraints:
        df = pd.DataFrame([parse_constraint(c) for c in raw_constraints])
        breakdown = df.groupby('template').size().rename('count').sort_values(ascending=False)
        print()
        display(breakdown.to_frame())
        print()
        display(df)
    else:
        print('(No constraints above support threshold)')



ROLE: bot
Cases: 1045
Distinct activities: 11
Computing discovery ...
Discovered 0 constraints
Saved DECLARE rules to: declare_models/declare_bot.json
(No constraints above support threshold)

ROLE: contributor
Cases: 109
Distinct activities: 1
Skipping: fewer than 2 distinct activities

ROLE: devops_engineer
Cases: 7
Distinct activities: 1
Skipping: fewer than 2 distinct activities

ROLE: feature_developer
Cases: 27
Distinct activities: 1
Skipping: fewer than 2 distinct activities

ROLE: issue_reporter
Cases: 1436
Distinct activities: 34
Computing discovery ...
Discovered 3 constraints
Saved DECLARE rules to: declare_models/declare_issue_reporter.json



,count
template,
Absence2,1
Exactly1,1
Existence1,1


,template,activity_a,activity_b
0,Existence1,closed,
1,Absence2,closed,
2,Exactly1,closed,



ROLE: maintainer
Cases: 837
Distinct activities: 1
Skipping: fewer than 2 distinct activities

ROLE: quality_engineer
Cases: 60
Distinct activities: 1
Skipping: fewer than 2 distinct activities

ROLE: technical_writer
Cases: 37
Distinct activities: 1
Skipping: fewer than 2 distinct activities


In [5]:
df_results = pd.DataFrame(discovery_rows).set_index('role').sort_index()
print('Discovery summary:')
display(df_results)

saved_files = sorted(OUTPUT_DIR.glob('declare_*.json'))
print(f"\nSaved DECLARE model files ({len(saved_files)}):")
for path in saved_files:
    size_kb = path.stat().st_size / 1024
    print(f'  - {path.name} ({size_kb:.2f} KB)')


Discovery summary:


,cases,distinct_activities,constraints,status,output_file
role,,,,,
bot,1045,11,0,saved,declare_models/declare_bot.json
contributor,109,1,0,skipped_low_activity_diversity,None
devops_engineer,7,1,0,skipped_low_activity_diversity,None
feature_developer,27,1,0,skipped_low_activity_diversity,None
issue_reporter,1436,34,3,saved,declare_models/declare_issue_reporter.json
maintainer,837,1,0,skipped_low_activity_diversity,None
quality_engineer,60,1,0,skipped_low_activity_diversity,None
technical_writer,37,1,0,skipped_low_activity_diversity,None



Saved DECLARE model files (2):
  - declare_bot.json (0.26 KB)
  - declare_issue_reporter.json (0.67 KB)
